# NB_lab_test_trends
Builds `adb_neeleshcaresync.gold.lab_test_trends` from the silver layer per the STTM (`sttm_silver_to_gold.xlsx`, sheet `lab_test_trends`).

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `silver.lab_results` grouped by `test_name`, year/month of `test_date`
* **Joins (LEFT, optional):** `silver.calendar` on `date_key = date_format(date_trunc('month', test_date), 'yyyyMMdd')`
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month` (month-over-month trend)
* **Filters:** none
* **Load type:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
# Catalog / schema configuration (STTM placeholder <catalog>)
catalog = "adb_neeleshcaresync"
silver = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per test type per month with month-over-month abnormal rate trend (STTM lab_test_trends)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                          AS year,
    month(test_date)                         AS month,
    COUNT(*)                                 AS total_tests,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)                                  AS test_name,
  CAST(w.year AS INT)                                          AS year,
  CAST(w.month AS INT)                                         AS month,
  CAST(c.month_name AS STRING)                                 AS month_name,
  CAST(c.quarter_name AS STRING)                               AS quarter_name,
  CAST(w.total_tests AS INT)                                   AS total_tests,
  CAST(w.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                           AS critical_test_count,
  w.abnormal_rate_pct                                          AS abnormal_rate_pct,
  w.critical_rate_pct                                          AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))         AS prev_month_abnormal_rate_pct,
  CAST(
    CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
         ELSE w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct
    END AS DECIMAL(6,2))                                       AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {silver}.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
""")
print(f"Created {target_table}")

In [0]:
# Return the gold table record count to the caller (e.g. job / dbutils.notebook.run)
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))